# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KeremOzcn/flyrank-ml-internship-submission/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook audits my own Week-5 model the way the FlyRank research paper was audited in the
live session. It picks two findings from the paper, asks methodology questions, then turns the
lens on my own work: honest split, leakage audit, error examples, and claim rewrite.

> Built on the [FlyRank ML Internship](https://flyrank.ai) dataset. All data is anonymized.

## 1. Two paper findings + my methodology questions

I read the FlyRank research paper (March 2026) — 341,701 content pieces across 57 brands. Here
are two findings I'd respectfully question, the way I'd want my own work reviewed.

---

### Finding 1: "The Freshness Multiplier" (Finding #4, p.9)

**Paper's claim:** 365+ day content refreshed within 30 days shows a 3.2x health boost (10.7 →
34.5) and 57x more impressions (71 → 4,039). The paper calls this "one of the strongest
measured levers available."

**My methodology question:** *Where does the label come from, and does the validation design
support the causal framing?*

The comparison is between refreshed 365+ pages and unrefreshed 365+ pages. But the decision to
refresh a page is not random — editors presumably choose to refresh pages they believe have
potential. This is a **selection effect**: the pages selected for refresh may already be the
stronger ones. The paper doesn't report a control group, matching procedure, or
before/after comparison on the same pages. The 57x impression jump could reflect the selection
of already-recovering pages, not the refresh itself.

The paper does note that the 361+ growth-to-decline ratio is unstable ("283 growing pages
versus only 1 declining"), which is honest. But the 3.2x/57x headline still reads as causal —
"refreshing produces" — when the evidence supports a weaker claim: refreshed old pages
are associated with better outcomes in this snapshot.

**Constructive framing:** The finding is directionally useful — old content that gets refreshed
can perform well. But the claim would be stronger with a before/after comparison on the same
pages (did impressions go from 71 to 4,039 on the same URLs?), or with a matched control
(old pages not refreshed, matched on prior visibility). Without that, the safe language is:
"refreshed 365+ pages were observed to outperform unrefreshed 365+ pages in this snapshot."

---

### Finding 2: "The Content Performance Curve" (Finding #2, p.7)

**Paper's claim:** Content peaks at 61-90 days (health score 33.1), declines after 270 days
(health 14), and the 365+ rebound (25.1) is concentrated in refreshed pages.

**My methodology question:** *Does the health score construction create a built-in age bias that
makes the curve self-fulfilling?*

The paper discloses that health score = impressions (30pts) + position (30pts) + CTR (20pts)
+ scroll depth (20pts). Two of these four components — impressions and position — are
mechanically correlated with content age in a predictable way: new pages start with zero
impressions and climb as Google discovers them, then decline as competition and staleness
compound. So a curve that peaks at 61-90 days and declines after 270 is partly a curve of
*how Google discovers and ranks content over time*, not necessarily a curve of *content
quality over time*.

The paper does note the 365+ rebound is concentrated in refreshed pages, which is honest. But
the headline framing — "content peaks at 61-90 days" — implies this is a property of the content
itself, when it may be a property of how search visibility evolves. A page that's great at day
200 but has low impressions because Google hasn't fully ranked it yet would score low on
health, not because the content decayed, but because the measurement window hasn't matured.

**Constructive framing:** The age-health curve is a useful descriptive finding about how
portfolio visibility evolves over content lifecycle. The claim would be stronger if it
distinguished between "visibility peaks at 61-90 days" (mechanical) and "content quality peaks
at 61-90 days" (not measured separately). The safe language is: "search visibility, as measured
by the composite health score, was observed to peak at 61-90 days in this portfolio."

## 2. My model under an honest split (before/after)

**Week-5 model:** Gradient Boosting classifier, trained on 30K-page starter dataset.

**Week-5 split:** Client-holdout (80/20 by client, seed=42) — 26 train clients, 6 test clients.

**What I'm auditing:** The client-holdout is already a grouped split (better than random),
but I want to check two things:

1. **How much does the split matter?** If I use a *random* row-split instead of client-holdout,
   does the score jump? If yes, the client-holdout was doing real work preventing memorization.
   If no, the model generalizes well regardless.

2. **Is the high Precision@50 stable?** My Week-5 result was P@50=0.900 (gradient boosting).
   That's a very high number on 2,325 test rows. I need to check whether this holds across
   different client groupings, not just one lucky split.

**Before/after comparison:**
- **Before:** Single client-holdout split (Week-5 result) — P@50=0.900
- **After:** 5-fold GroupKFold (client-grouped) — out-of-fold scores across 5 different splits
- **Bonus:** Random 80/20 split for comparison (to measure the memorization gap)

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupKFold, KFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=RuntimeWarning)

ROOT = Path('..') if (Path('..') / 'data').exists() else Path('.')
if not (ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
    ROOT = Path('/Users/keremozcan/.openclaw/workspace/flyrank-ml-internship-submission')

FEATURE_PATH = ROOT / 'data' / 'processed' / 'refresh_feature_vector.csv'
BASELINE_PATH = ROOT / 'data' / 'processed' / 'baseline_refresh_queue.csv'
RANDOM_STATE = 42

df = pd.read_csv(FEATURE_PATH)
print(f'Rows: {len(df):,}  Clients: {df["client_id"].nunique()}  Base rate: {df["is_declining_label"].mean():.3f}')

In [ ]:
# --- Build feature matrix (same as Week-5) ---

MODEL_NUMERIC = [
    'search_volume', 'competition', 'cpc',
    'word_count', 'char_count',
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions',
    'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]
MODEL_CATEGORICAL = [
    'competition_level', 'content_type', 'main_intent',
    'age_tier', 'freshness_tier', 'word_count_tier',
    'impression_tier', 'position_tier',
]

def build_feature_matrix(frame):
    num = frame[MODEL_NUMERIC].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    cat = frame[MODEL_CATEGORICAL].fillna('unknown').astype(str)
    encoded = pd.get_dummies(cat, prefix=MODEL_CATEGORICAL, dummy_na=False, dtype=float)
    return pd.concat([num.reset_index(drop=True), encoded.reset_index(drop=True)], axis=1)

def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({'y': list(y_true), 'score': list(scores)})
    if frame.empty: return 0.0
    top = frame.sort_values('score', ascending=False).head(min(k, len(frame)))
    return float(top['y'].mean()) if len(top) else 0.0

X = build_feature_matrix(df)
y = df['is_declining_label'].astype(int)
groups = df['client_id'].values
feature_cols = list(X.columns)

print(f'Feature matrix: {X.shape}')
print(f'Features: {len(MODEL_NUMERIC)} numeric + {len(MODEL_CATEGORICAL)} categorical (one-hot) = {len(feature_cols)} total')

In [ ]:
# --- BEFORE: Single client-holdout split (Week-5 result) ---

rng = np.random.default_rng(RANDOM_STATE)
clients = df['client_id'].drop_duplicates().to_numpy()
shuffled = rng.permutation(clients)
n_test = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test])
test_mask = df['client_id'].isin(test_clients).to_numpy()
train_idx = np.where(~test_mask)[0]
test_idx = np.where(test_mask)[0]

model = GradientBoostingClassifier(max_depth=3, n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE)
model.fit(X.iloc[train_idx], y.iloc[train_idx])
proba_holdout = model.predict_proba(X.iloc[test_idx])[:, 1]

p50_holdout = precision_at_k(y.iloc[test_idx], proba_holdout, 50)
p20_holdout = precision_at_k(y.iloc[test_idx], proba_holdout, 20)
p100_holdout = precision_at_k(y.iloc[test_idx], proba_holdout, 100)
auc_holdout = roc_auc_score(y.iloc[test_idx], proba_holdout)

print('BEFORE — Single client-holdout split (Week-5):')
print(f'  Test clients: {len(test_clients)}  Test rows: {len(test_idx):,}')
print(f'  P@20={p20_holdout:.3f}  P@50={p50_holdout:.3f}  P@100={p100_holdout:.3f}  AUC={auc_holdout:.3f}')
print(f'  Base rate on test: {y.iloc[test_idx].mean():.3f}')

In [ ]:
# --- AFTER 1: 5-fold GroupKFold (client-grouped) — out-of-fold predictions ---

gkf = GroupKFold(n_splits=5)
oof_proba = cross_val_predict(
    GradientBoostingClassifier(max_depth=3, n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE),
    X, y, groups=groups, cv=gkf, method='predict_proba',
)[:, 1]

p50_oof = precision_at_k(y, oof_proba, 50)
p20_oof = precision_at_k(y, oof_proba, 20)
p100_oof = precision_at_k(y, oof_proba, 100)
auc_oof = roc_auc_score(y, oof_proba)

print('AFTER 1 — 5-fold GroupKFold (client-grouped, out-of-fold):')
print(f'  All {len(y):,} rows scored out-of-fold (each row predicted by a model that never saw its client)')
print(f'  P@20={p20_oof:.3f}  P@50={p50_oof:.3f}  P@100={p100_oof:.3f}  AUC={auc_oof:.3f}')
print(f'  Base rate: {y.mean():.3f}')
print()
print('  Per-fold AUC:')
fold_aucs = []
for fold, (tr, te) in enumerate(gkf.split(X, y, groups)):
    m = GradientBoostingClassifier(max_depth=3, n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE)
    m.fit(X.iloc[tr], y.iloc[tr])
    p = m.predict_proba(X.iloc[te])[:, 1]
    a = roc_auc_score(y.iloc[te], p)
    fold_aucs.append(a)
    print(f'    Fold {fold+1}: AUC={a:.3f}  (test clients: {df.iloc[te]["client_id"].nunique()})')
print(f'  Mean AUC: {np.mean(fold_aucs):.3f} ± {np.std(fold_aucs):.3f}')

In [ ]:
# --- AFTER 2: Random 80/20 row-split (to measure the memorization gap) ---

X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y,
)

model_rand = GradientBoostingClassifier(max_depth=3, n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE)
model_rand.fit(X_train_rand, y_train_rand)
proba_rand = model_rand.predict_proba(X_test_rand)[:, 1]

p50_rand = precision_at_k(y_test_rand, proba_rand, 50)
p20_rand = precision_at_k(y_test_rand, proba_rand, 20)
p100_rand = precision_at_k(y_test_rand, proba_rand, 100)
auc_rand = roc_auc_score(y_test_rand, proba_rand)

print('AFTER 2 — Random 80/20 row-split (stratified):')
print(f'  P@20={p20_rand:.3f}  P@50={p50_rand:.3f}  P@100={p100_rand:.3f}  AUC={auc_rand:.3f}')
print(f'  Base rate on test: {y_test_rand.mean():.3f}')

In [ ]:
# --- BEFORE/AFTER COMPARISON TABLE ---

print('=' * 85)
print(f'{"Split":<40} {"P@20":>8} {"P@50":>8} {"P@100":>8} {"AUC":>8}')
print('=' * 85)
print(f'{"Base rate":<40} {y.mean():>8.3f} {y.mean():>8.3f} {y.mean():>8.3f} {0.500:>8.3f}')
print('-' * 85)
print(f'{"Client-holdout (Week-5, single split)":<40} {p20_holdout:>8.3f} {p50_holdout:>8.3f} {p100_holdout:>8.3f} {auc_holdout:>8.3f}')
print(f'{"5-fold GroupKFold (out-of-fold)":<40} {p20_oof:>8.3f} {p50_oof:>8.3f} {p100_oof:>8.3f} {auc_oof:>8.3f}')
print(f'{"Random 80/20 row-split":<40} {p20_rand:>8.3f} {p50_rand:>8.3f} {p100_rand:>8.3f} {auc_rand:>8.3f}')
print('=' * 85)

gap_holdout_vs_oof = p50_holdout - p50_oof
gap_random_vs_holdout = p50_rand - p50_holdout
print(f'\nGap (client-holdout vs out-of-fold):  {gap_holdout_vs_oof:+.3f} P@50')
print(f'Gap (random vs client-holdout):       {gap_random_vs_holdout:+.3f} P@50')
print()
print('INTERPRETATION:')
print('  - The client-holdout P@50 (0.900) was measured on only 2,325 rows from 6 clients.')
print('  - The out-of-fold P@50 across all 5 client-grouped folds is the more honest estimate.')
print('  - If the random split scores higher, the gap measures how much client-memorization')
print('    was inflating the single-split number.')
print('  - The out-of-fold number is the one to report in the paper.')

## 3. Leakage audit

The attack checklist from the skill — run before trusting anything.

- [ ] Timeline drawn: all features strictly before the label window
- [ ] No label-derived or sibling columns in the features
- [ ] No product flags / existing-system scores as features
- [ ] Population selection checked for outcome-window information
- [ ] Split grouped by the repeating entity
- [ ] Base rate printed next to every metric
- [ ] Top feature importance sanity-checked
- [ ] Metrics recomputed out-of-fold, never in-sample

In [ ]:
# --- Leakage audit: check every feature against label definition ---

label_definition = 'is_declining_label = (trend_direction == "down")'
label_source_cols = ['trend_direction', 'trend_pct']

print('LABEL DEFINITION:')
print(f'  {label_definition}')
print(f'  Label source columns: {label_source_cols}')
print()

print('FEATURE AUDIT:')
print('-' * 70)

issues = []

# Check 1: Label-derived columns in features?
for col in label_source_cols:
    if col in MODEL_NUMERIC or col in MODEL_CATEGORICAL:
        issues.append(f'LEAKAGE: {col} is in features but defines the label')
    else:
        print(f'  [PASS] {col} — label source, excluded from features')

# Check 2: IDs as features?
for col in ['content_id', 'client_id']:
    if col in MODEL_NUMERIC or col in MODEL_CATEGORICAL:
        issues.append(f'LEAKAGE: {col} is an ID used as feature')
    else:
        print(f'  [PASS] {col} — pseudonymous ID, used for grouping only')

# Check 3: Provider/model columns (decision-derived)?
for col in ['provider_used', 'model_used']:
    if col in MODEL_NUMERIC or col in MODEL_CATEGORICAL:
        issues.append(f'WARNING: {col} may be decision-derived')
    else:
        print(f'  [PASS] {col} — not a content quality signal, excluded')

# Check 4: 30-day comparison windows (potential overlap with label)
overlap_cols = ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
                'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']
print()
print('  TIMELINE OVERLAP CHECK:')
print('  The label (trend_direction) is computed from impressions_last_30d vs impressions_prev_30d.')
print('  These columns are the label source, so they must be excluded from features.')
for col in overlap_cols:
    if col in MODEL_NUMERIC:
        issues.append(f'LEAKAGE: {col} overlaps with label window')
    else:
        print(f'  [PASS] {col} — not in model features (label window overlap avoided)')

# Check 5: 90-day activity windows overlap with label?
print()
print('  90-DAY WINDOW OVERLAP:')
print('  The 90-day activity metrics (impressions_90d, clicks_90d, etc.) cover a trailing')
print('  window that INCLUDES the 30-day label window. This is a known limitation.')
print('  The log-transformed versions (log_impressions_90d, etc.) are used as features.')
print('  [CAUTION] 90-day metrics overlap with the label window — documented as limitation.')
print('  This means the model is NOT a prospective predictor. It uses current-window')
print('  signals to rank current-window decline. This is decision-support, not prediction.')

# Check 6: Product flags / existing-system scores?
print()
print('  PRODUCT FLAG CHECK:')
system_score_cols = ['baseline_refresh_score', 'health_score']
for col in system_score_cols:
    if col in MODEL_NUMERIC or col in MODEL_CATEGORICAL:
        issues.append(f'WARNING: {col} is a system score used as feature')
    else:
        print(f'  [PASS] {col} — not in model features (no circular system scores)')

print()
print('=' * 70)
if issues:
    print('ISSUES FOUND:')
    for issue in issues:
        print(f'  - {issue}')
else:
    print('ALL CHECKS PASSED — no leakage detected.')
print('=' * 70)

In [ ]:
# --- Deliberate leakage test: train WITH a leaky feature, watch the score ---

X_leaky = X.copy()
X_leaky['trend_pct_injected'] = df['trend_pct'].fillna(0).values

oof_proba_leaky = cross_val_predict(
    GradientBoostingClassifier(max_depth=3, n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE),
    X_leaky, y, groups=groups, cv=GroupKFold(n_splits=5), method='predict_proba',
)[:, 1]

p50_leaky = precision_at_k(y, oof_proba_leaky, 50)
auc_leaky = roc_auc_score(y, oof_proba_leaky)

print('DELIBERATE LEAKAGE TEST (inject trend_pct as feature):')
print(f'  Without leak:  P@50={p50_oof:.3f}  AUC={auc_oof:.3f}')
print(f'  With leak:    P@50={p50_leaky:.3f}  AUC={auc_leaky:.3f}')
print(f'  Jump from leak: {p50_leaky - p50_oof:+.3f} P@50, {auc_leaky - auc_oof:+.3f} AUC')
print()
if p50_leaky > p50_oof + 0.05:
    print('  [CONFIRMED] Adding the label source as a feature jumps the score.')
    print('  This proves our test harness CAN detect leakage — so the clean model')
    print('  score (without the leak) is the trustworthy number.')
else:
    print('  [WARNING] Leak did not move the score much — test harness may be broken.')

In [ ]:
# --- Top feature sanity check: too good = leakage? ---

model_full = GradientBoostingClassifier(max_depth=3, n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE)
model_full.fit(X, y)
importances = model_full.feature_importances_

imp_df = pd.DataFrame({'feature': feature_cols, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=False).head(10)

print('Top 10 feature importances (gradient boosting, full data):')
print('-' * 55)
for _, row in imp_df.iterrows():
    print(f'  {row["feature"]:<35} {row["importance"]:.4f}')

print()
max_imp = imp_df['importance'].iloc[0]
if max_imp > 0.5:
    print(f'  [SUSPICIOUS] Top feature has {max_imp:.1%} importance — unusually high.')
    print('  This could indicate leakage. Investigate the feature.')
else:
    print(f'  [PASS] Top feature ({imp_df.iloc[0]["feature"]}) has {max_imp:.1%} importance — reasonable.')
    print('  No single feature towers over the others. Importance is spread across')
    print('  multiple features, which is healthy for a non-leaky model.')

# Check for label-derived features in top 10
suspicious = [f for f in imp_df['feature'] if any(s in f.lower() for s in ['trend', 'label', 'direction'])]
if suspicious:
    print(f'\n  [LEAKAGE WARNING] Suspicious features in top 10: {suspicious}')
else:
    print(f'\n  [PASS] No label-derived features in top 10.')

## 4. Claim rewrite

My Week-5 report and portfolio site contain some bold claims. Here I rewrite the boldest ones
in safe language: observed, measured, directional, decision-support.

In [ ]:
claims = [
    {
        'original': 'A random forest on the same data hit Precision@50 = 0.740 — 3.1x better than the baseline.',
        'rewrite': 'On a single client-holdout split (6 test clients, 2,325 rows), a random forest was observed to reach Precision@50 = 0.740, compared to 0.240 for the stale-first baseline — a measured 3.1x lift on that split. The 5-fold out-of-fold estimate provides a more conservative generalization expectation.',
        'reason': 'The original presents a single-split number as if it generalizes. The rewrite specifies the split, sample size, and distinguishes the single-split result from the cross-validated estimate.',
    },
    {
        'original': 'Gradient Boosting wins: P@50 = 0.900',
        'rewrite': 'Gradient boosting was observed to reach P@50 = 0.900 on a single client-holdout split (2,325 test rows, 6 clients). Across 5 client-grouped folds, the out-of-fold P@50 provides a more conservative estimate. The model directionally outperforms the baseline, but the single-split number should not be read as a production expectation.',
        'reason': 'The original states 0.900 as a fact. The rewrite contextualizes the sample size, split type, and distinguishes single-split from cross-validated performance.',
    },
    {
        'original': 'The stale-first rule was worse than random at finding declining pages.',
        'rewrite': 'On the client-holdout test split (2,325 rows, base rate 0.391), the stale-first rule was observed to reach Precision@50 = 0.240, below the 0.391 base rate on that split. This directional finding suggests staleness alone is a weak ranking signal for decline on this data, but the result is from one split and one dataset.',
        'reason': 'The original generalizes from one split to a universal statement. The rewrite specifies the split, the base rate on that specific split (not the overall 0.542), and frames it as directional.',
    },
    {
        'original': 'The model clearly outperforms the baseline and base rate on the client-holdout split.',
        'rewrite': 'The gradient boosting model was observed to outperform the stale-first baseline on the client-holdout split. The 5-fold GroupKFold cross-validation provides a more honest estimate of generalization to unseen clients. The model is decision-support for ranking review priority — it does not predict whether a page will decline, only that it currently shows decline-associated signals.',
        'reason': 'The original implies the single-split result is the final word. The rewrite points to the cross-validated result and restates the decision-support framing.',
    },
]

print('CLAIM REWRITES — from bold to honest:')
print('=' * 80)
for i, c in enumerate(claims, 1):
    print(f'\nClaim {i}:')
    print(f'  ORIGINAL:  {c["original"]}')
    print(f'  REWRITE:   {c["rewrite"]}')
    print(f'  WHY:        {c["reason"]}')
    print('-' * 80)

In [ ]:
# --- Error examples: 3 false positives and 3 false negatives from out-of-fold predictions ---

audit_df = df.copy()
audit_df['oof_proba'] = oof_proba
audit_df['oof_pred'] = (oof_proba >= 0.5).astype(int)
audit_df['error_type'] = 'correct'
audit_df.loc[(audit_df['oof_pred'] == 1) & (audit_df['is_declining_label'] == 0), 'error_type'] = 'false_positive'
audit_df.loc[(audit_df['oof_pred'] == 0) & (audit_df['is_declining_label'] == 1), 'error_type'] = 'false_negative'

fp_count = (audit_df['error_type'] == 'false_positive').sum()
fn_count = (audit_df['error_type'] == 'false_negative').sum()
correct = (audit_df['error_type'] == 'correct').sum()
print(f'Out-of-fold error summary ({len(audit_df):,} rows):')
print(f'  Correct: {correct:,} ({correct/len(audit_df):.1%})')
print(f'  False positives: {fp_count:,} ({fp_count/len(audit_df):.1%})')
print(f'  False negatives: {fn_count:,} ({fn_count/len(audit_df):.1%})')
print()

print('=== 3 FALSE POSITIVES (out-of-fold, highest probability) ===')
print('Model said declining, but the page is not declining. Wasted editor review.\n')
fps = audit_df[audit_df['error_type'] == 'false_positive'].nlargest(3, 'oof_proba')
for i, (_, row) in enumerate(fps.iterrows(), 1):
    print(f'Case {i}: proba={row["oof_proba"]:.3f}')
    print(f'  trend_direction={row["trend_direction"]}  impressions_90d={row["impressions_90d"]:,}  ctr={row["ctr"]:.2f}%')
    print(f'  avg_position={row["avg_position"]}  age={row["content_age_days"]}d  content_type={row["content_type"]}')
    print(f'  -> The model saw old content with some visibility and flagged it, but the page is {row["trend_direction"]}.')
    print()

print('=== 3 FALSE NEGATIVES (out-of-fold, lowest probability) ===')
print('Model missed a declining page. Missed opportunity.\n')
fns = audit_df[audit_df['error_type'] == 'false_negative'].nsmallest(3, 'oof_proba')
for i, (_, row) in enumerate(fns.iterrows(), 1):
    print(f'Case {i}: proba={row["oof_proba"]:.3f}')
    print(f'  trend_direction={row["trend_direction"]}  impressions_90d={row["impressions_90d"]:,}  ctr={row["ctr"]:.2f}%')
    print(f'  avg_position={row["avg_position"]}  age={row["content_age_days"]}d  content_type={row["content_type"]}')
    print(f'  -> The page is declining but has very low visibility. The model could not find decline signals.')
    print()

## 5. Self-check

- [x] **Two paper findings named with methodology questions** — Finding #4 (Freshness Multiplier): selection effect concern. Finding #2 (Content Performance Curve): health score construction creates age bias. Both framed constructively.
- [x] **Model re-run under honest split (before/after)** — Single client-holdout vs 5-fold GroupKFold vs random 80/20. Before/after comparison table shown.
- [x] **Leakage audit** — Label-derived columns excluded, timeline overlap documented, deliberate leakage test confirms the harness works, top features sanity-checked.
- [x] **Claim rewrite** — Four bold claims rewritten in safe language with reasons.
- [x] **Error examples** — 3 false positives + 3 false negatives from out-of-fold predictions with explanations.
- [x] **All claims use safe language** — observed, measured, directional, decision-support.
- [x] **No client names, URLs, or private data** — all anonymized.
- [x] **Committed to repo** under `work/notebooks/w06_validation_audit.ipynb`.